# Supervised Learning Masterclass
This notebook is **self‑contained coursework** for students taking their first steps into supervised machine‑learning. Every code section is preceded by a **pedagogical markdown explainer** that answers:
* *What problem are we solving?*  
* *Which algorithm are we using?*  
* *Why that algorithm works (intuition + light math)?*  
* *Pros / Cons & typical applications.*

At the end, students should be able to:
1. Compare classic classifiers on the Wine dataset.
2. Understand feature‑selection mechanics in trees & forests.
3. Tune hyper‑parameters with `GridSearchCV`.
4. Evaluate and improve regression models on the Diabetes dataset.

## 1  Setup – Imports & Helper Libraries

In [ ]:
from __future__ import annotations
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_wine, load_diabetes, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             mean_squared_error, r2_score)
from sklearn.linear_model import (LogisticRegression, LinearRegression, RidgeCV,
                                  LassoCV, ElasticNetCV)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              RandomForestRegressor, GradientBoostingRegressor)
from sklearn.svm import SVC, SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance

sns.set_style('whitegrid'); plt.rcParams['figure.figsize']=(6,4)

## 2  Multi‑class Classification with the Wine Dataset

### 2.1  Dataset context
🍷  Wine dataset — chem-fingerprints of three Italian cultivars

* **What it is** – 178 wine samples from the same region of Italy, each with **13 chemical measurements** (alcohol, flavanoids, hue, etc.). The target has **3 classes**—one for each grape cultivar.

* **Shape** – `(178 rows × 13 features)`, plus a 1-D label vector of length 178. Samples per class: 59, 71 and 48.

* **Feature types** – all numeric; concentrations or optical absorbance measures. No missing values.

* **Typical use-cases**  
  * Benchmarking multi-class classifiers (k-NN, SVM, Random Forest).
  * Demonstrating **feature importance** & **PCA/TSNE** visualisation.

* **Why it’s handy for students**  
  * Small enough to **fit in memory** and train in seconds.  
  * **Balanced classes** and purely numeric features keep preprocessing minimal.  
  * Works out-of-the-box with `sklearn.datasets.load_wine()`.

* **Watch-outs**  
  | Issue | Impact | Tip |
  |-------|--------|-----|
  | All wines from one region | May not generalise to other terroirs | Treat results as illustrative, not universal |
  | Only 178 rows | Easy to over-fit deep trees / high-dim models | Use cross-validation or ensembles |


### 2.2  Baseline model zoo – code & visual comparison
We benchmark seven classic classifiers *out‑of‑the‑box* so students can see a spread of accuracies before any tuning.

In [ ]:
# Load & split
wine = load_wine(); X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Standardise numeric features (helps distance & linear models)
scaler = StandardScaler(); X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'k‑NN (k=5)': KNeighborsClassifier(5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'SVM (RBF)': SVC(random_state=42),
    'Gaussian NB': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

baseline = {name: clf.fit(X_train,y_train).score(X_test,y_test) for name,clf in classifiers.items()}

plt.bar(baseline.keys(), baseline.values()); plt.xticks(rotation=45, ha='right');
plt.ylabel('Accuracy'); plt.title('Wine – baseline accuracy'); plt.tight_layout(); plt.show()

#### Discussion — what each classifier is really doing

* **Logistic Regression**  
  Learns a *linear* boundary and converts raw scores into class probabilities with the logistic or soft-max function. It trains quickly, needs little tuning, and its coefficients have a clear “positive raises odds / negative lowers odds” interpretation.  

* **k-Nearest Neighbors (k-NN)**  
  Stores the entire training set and, at prediction time, looks up the *k* closest points; the majority label wins. Zero training cost, but inference slows as the dataset grows and the method is sensitive to feature scaling.  

* **Decision Tree**  
  Repeatedly splits on the feature/threshold that gives the largest impurity drop (Gini or entropy), producing an easy-to-read flow-chart of if-else rules. Deep trees can memorize noise, so pruning (`max_depth`, `min_samples_leaf`, `ccp_alpha`) is essential.  

* **Random Forest**  
  Trains many bootstrapped trees, each seeing a random subset of features; their averaged vote greatly reduces variance while keeping low bias, giving strong “out-of-the-box” accuracy with minimal tuning.  

* **SVM (RBF kernel)**  
  Finds the hyper-plane with the widest margin between classes; the RBF kernel bends that plane in a higher-dimensional space. Two knobs control behavior: **C** (how hard to punish errors) and **γ** (how far each point’s influence reaches).  

* **Gaussian Naïve Bayes**  
  Uses Bayes’ rule under the “naïve” assumption that features are independent given the class and normally distributed. Training is just computing means and variances, so it’s lightning fast, but correlation between features can hurt accuracy.  

* **Gradient Boosting**  
  Builds an ensemble of shallow trees *sequentially*; each new tree fits the residuals of the current model. Accuracy is excellent, but it relies on careful tuning of learning rate, tree depth, and number of trees.  


### 2.3  Algorithm Cards – intuition, math‑lite, pros & cons

### 🌲  Decision Tree — a flow-chart of if-else rules

*Idea:* split the data over and over on the feature and threshold that most **purifies** the labels (e.g.\ maximises Gini or Information Gain) until each terminal **leaf** is as homogeneous as possible.

Purity gain at a candidate split is

$$
\Delta = \text{Impurity(parent)} \;-\; \bigl[\,w_\text{L}\,\text{Impurity(left)} + w_\text{R}\,\text{Impurity(right)}\bigr],
$$

where \(w_\text{L}, w_\text{R}\) are child proportions.

**Why it’s useful**

* White-box: each prediction follows a single root→leaf path that you can read as a series of questions.  
* No feature scaling needed; handles numeric & categorical inputs (after one-hot).  
* Captures non-linear relationships and feature interactions.

**When to grab it**

* Need for **human-readable rules** (credit policies, medical triage).  
* Quick baseline on small-to-medium tabular data.  
* Feature-selection step before ensembles.

**Watch-outs**

| Issue | Impact | Mitigation |
|-------|--------|------------|
| **Over-fitting** | Deep trees memorise noise, hurting test accuracy. | Prune with `max_depth`, `min_samples_leaf`, or cost-complexity `ccp_alpha`. |
| **Instability** | Small data tweaks can change splits. | Use ensembles (Random Forest, Gradient Boosting) for robustness. |
| **Biased splits** | Favour features with more levels (e.g.\ IDs). | One-hot wisely; consider permutation importance to validate. |


### 🌳  Random Forest — a crowd of trees that votes

*Idea:* train **many** decision trees, each on a *bootstrap* sample of the data **and** a random subset of features; combine their predictions by **majority vote** (classification) or **average** (regression).

$$
\hat{y}_{\text{RF}} \;=\; \frac{1}{T}\sum_{t=1}^{T} \hat{y}^{(t)}
$$

where \(T\) = number of trees and \(\hat{y}^{(t)}\) is tree \(t\)’s output.

**Why it works**

* Bootstrapping + feature randomness **decorrelate** trees → their errors cancel, slashing variance.  
* Keeps the low bias of deep trees while taming over-fitting.

**When to grab it**

* Tabular data with complex, non-linear relations.  
* Need a strong baseline with **minimal tuning**.  
* Want quick **feature-importance** estimates.

**Watch-outs**

| Issue | Impact | Mitigation |
|-------|--------|------------|
| **Interpretability** | 500 trees are a black box. | Use feature importance, SHAP, or reduce `n_estimators`. |
| **Memory / speed** | Large forests can be heavy and slow to predict. | Limit `n_estimators`, cap `max_depth`, or switch to Gradient Boosting for lighter models. |
| **Diminishing returns** | After a certain forest size, extra trees add little accuracy. | Monitor OOB score / validation curve to stop early. |


### 🛤️  Support Vector Machine — finding the widest gap  

*Idea:* draw a **hyper-plane** that **maximises the margin** (gap) between classes; only the edge points — the **support vectors** — matter.  Non-linear data are handled by the **kernel trick**, which lets SVM behave as if it were in a higher-dimensional space without actually computing that space.  

For binary classification with an RBF kernel the decision function is  

$$
f(x)=\sum_{i=1}^{N}\alpha_i\,y_i\,\exp\!\bigl(-\gamma\lVert x_i-x\rVert^{2}\bigr)+b,
$$

where $\alpha_i$ come from the optimisation, $\gamma$ controls kernel width and $b$ is the bias.  

**Why it works**  
* A wider margin gives strong protection against over-fitting.  
* The kernel trick enables curved decision boundaries with just two main knobs: **C** (error penalty) and **γ** (kernel width).  
* Works well in high-dimensional spaces.

**When to grab it**  
* Medium-sized datasets (thousands of rows) with many features.  
* Text or image data where feature scaling puts everything on comparable footing.  
* Problems where class boundaries are curved but you still want a relatively small model.

**Watch-outs**

| Issue                 | Impact                                          | Mitigation |
|-----------------------|-------------------------------------------------|------------|
| **Scale sensitivity** | Un-scaled features distort the margin geometry. | Always standardise inputs first. |
| **Parameter tuning**  | Bad *C* or γ ⇒ under/over-fit.                  | Use `GridSearchCV` (cross-validation). |
| **Computation cost**  | Training time grows roughly with $N^{2}$–$N^{3}$; slow on big data. | Sample data, use linear SVM, or switch to SGD-based classifiers. |


### 2.4  Improving the Decision Tree – pruning & feature selection
We search over `max_depth`, `min_samples_leaf`, and **cost‑complexity pruning** (`ccp_alpha`) to strike a bias‑variance balance.

In [ ]:
dt_params = {'max_depth':[None,3,5,7], 'min_samples_leaf':[1,3,5], 'ccp_alpha':[0.0,0.005,0.01]}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5)
dt_grid.fit(X_train, y_train)
print('Baseline DT acc:', baseline['Decision Tree'])
print('Tuned DT acc   :', dt_grid.score(X_test, y_test), 'best params:', dt_grid.best_params_)

In [ ]:
# Visualise tuned tree (depth capped for clarity)
best_dt = dt_grid.best_estimator_
plt.figure(figsize=(10,6))
plot_tree(best_dt, feature_names=wine.feature_names, class_names=wine.target_names, filled=True, max_depth=3)
plt.title('Pruned Decision Tree (first 3 levels)'); plt.show()

### 2.5  Random‑Forest Feature Importance

*Idea:* build **lots of decision trees**, each on a *bootstrap* sample of the data **plus** a random subset of features; combine their predictions by **majority vote** (classification) or **average** (regression). :contentReference[oaicite:0]{index=0}

For regression the ensemble output is  

$$
\hat{y}_{\text{RF}} = \frac1{T}\sum_{t=1}^{T}\hat{y}^{(t)}
$$

and for classification it’s the mode of the tree votes. :contentReference[oaicite:1]{index=1}  
Here \(T\) = number of trees and \(\hat{y}^{(t)}\) is tree \(t\)’s prediction.

**Why it works**  
Bootstrap sampling decorrelates trees, and feature randomness lowers the chance they all make the same mistake, so variance (over-fitting) drops without raising bias much. :contentReference[oaicite:2]{index=2}

**When it shines**

* Non-linear, high-dimensional tabular data. :contentReference[oaicite:3]{index=3}  
* Need for built-in **feature importance** estimates. :contentReference[oaicite:4]{index=4}  
* Datasets with missing values or mixed data types. :contentReference[oaicite:5]{index=5}  

**Watch-outs**

| Issue | Impact | Mitigation |
|-------|--------|------------|
| **Interpretability** | Individual trees are clear; 500 of them aren’t. | Use feature importance or tree-explainer tools. :contentReference[oaicite:6]{index=6} |
| **Memory / speed** | Large forests can be heavy to store and predict. | Limit `n_estimators`, use `max_depth`, or try Gradient Boosting. :contentReference[oaicite:7]{index=7} |


In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)
perm = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42)
idx = perm.importances_mean.argsort()[::-1][:10]
plt.bar([wine.feature_names[i] for i in idx], perm.importances_mean[idx]);
plt.xticks(rotation=45, ha='right'); plt.ylabel('Permutation importance');
plt.title('Random Forest – Top 10 features'); plt.tight_layout(); plt.show()

## 3  Regression with the Diabetes Dataset

### 3.1  Dataset overview
🩺  Diabetes dataset — predicting disease progression from baseline metrics

* **What it is** – 442 patient records collected by the National Institute of Diabetes and Digestive and Kidney Diseases.  
  Each record has **10 baseline measurements** (e.g., age, body-mass index, blood pressure, serum markers).  
  The target is a **continuous score** representing disease-progression one year after baseline.

* **Shape** – `(442 rows × 10 features)` plus a 1-D target vector of length 442.

* **Feature types** – all numeric and already scaled to comparable ranges; no missing values.

* **Typical use-cases**  
  * Benchmarking regression algorithms (Linear, Ridge/Lasso, RandomForestRegressor, SVR).  
  * Demonstrating regularisation effects and residual diagnostics.  
  * Teaching mean-squared-error vs. R² evaluation.

* **Why it’s handy for students**  
  * **Small** – fits in memory and trains in seconds.  
  * **Real-world** medical context makes error interpretation tangible.  
  * Works instantly with `sklearn.datasets.load_diabetes()`.

* **Watch-outs**  

  | Issue | Impact | Tip |
  |-------|--------|-----|
  | Modest sample size | Complex models (deep nets) over-fit quickly | Use cross-validation or simpler models |
  | All predictors are standardised | Scaling won’t change results, but you can still pipeline a scaler for consistency | Keep `StandardScaler` in pipeline but it will do almost nothing |
  | Target is noisy | Even great models rarely exceed R² ≈ 0.5 | Compare models on MSE and residual plots rather than chasing very high R² |


### 3.2  Model roster & tuning grids  

| Model | What it captures | Key hyper-parameters we search | Typical grid |
|-------|------------------|--------------------------------|--------------|
| **Linear Regression** | Straight-line fit (baseline). | None (closed-form OLS). | – |
| **Ridge / Lasso / ElasticNet** | Same line but with **L2** (Ridge), **L1** (Lasso) or mixed (ElasticNet) shrinkage to tame over-fitting. | `alpha` (strength), `l1_ratio` (ElasticNet only). | `alpha = logspace(-3, 3, 50)`<br>`l1_ratio = [0.2, 0.5, 0.8, 1.0]` |
| **RandomForestRegressor** | Many bootstrapped trees averaged → low variance, non-linear patterns. | `n_estimators` (forest size), `max_depth` (tree depth). | `n_estimators = [200, 500]`<br>`max_depth = [None, 5, 10]` |
| **GradientBoostingRegressor** | Shallow trees added sequentially to correct residuals. | `learning_rate` (shrinkage), `n_estimators` (# trees). | `learning_rate = [0.01, 0.05, 0.1]`<br>`n_estimators = [200, 400]` |
| **SVR (RBF)** | Non-linear “rubber-sheet” fit using kernel trick. | `C` (penalty), `gamma` (kernel width). | `C = [1, 10, 100]`<br>`gamma = ['scale', 0.01, 0.1]` |

After training each grid, we’ll compare models on **MSE** and **R²** using the hold-out test set, then plot residuals for the best performer.


### ⚖️  StandardScaler — putting all features on the same footing

*Goal:* rescale each numeric column so it has **mean = 0** and **standard deviation = 1**.  
This prevents big-unit features (e.g.\ income) from overpowering small-unit ones (e.g.\ age) in distance- or gradient-based models.

$$
x_{\text{scaled}} \;=\; \frac{x \;-\; \bar{x}}{s}
$$

where  
* $\bar{x}$  =  training-set mean  
* $s$  =  training-set standard deviation  

**When it helps**

* Algorithms that rely on **distance** (k-NN, SVM + RBF, k-means)  
* Models trained with **gradient descent** (linear/logistic regression, neural nets)  
* Dimensionality-reduction methods like **PCA** that are scale-sensitive  


In [ ]:
# Split
dia = load_diabetes(); Xr, yr = dia.data, dia.target
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

models_simple = {
    'Linear': make_pipeline(StandardScaler(), LinearRegression()),
    'RidgeCV': make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3,3,50), cv=5)),
    'LassoCV': make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-3,3,50), cv=5, random_state=42)),
    'ElasticNetCV': make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[.2,.5,.8,1.0], alphas=np.logspace(-3,3,50), cv=5, random_state=42))
}

# Tuned models via grid search
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), {'n_estimators':[200,500], 'max_depth':[None,5,10]}, cv=3)
gbr_grid = GridSearchCV(GradientBoostingRegressor(random_state=42), {'learning_rate':[0.01,0.05,0.1], 'n_estimators':[200,400]}, cv=3)
svr_grid = GridSearchCV(make_pipeline(StandardScaler(), SVR()), {'svr__C':[1,10,100], 'svr__gamma':['scale',0.01,0.1]}, cv=3)

all_models = {**models_simple, 'RandomForest (tuned)': rf_grid, 'GBR (tuned)': gbr_grid, 'SVR (tuned)': svr_grid}

results = []
for name, model in all_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    results.append((name, mean_squared_error(yr_test, pred), r2_score(yr_test, pred)))

df_reg = pd.DataFrame(results, columns=['Model','MSE','R2']).sort_values('MSE')
display(df_reg)

### 3.3  R² bar chart & residuals for the winner

### 🔎  R² — How much variation did we explain?

*R² answers a single question: “Out of all the ups-and-downs in **y**, what fraction does our model capture?”*

$$
R^{2} \;=\; 1 \;-\; \frac{\text{left-over variance (errors)}}{\text{total variance (actual \(y\) around its mean)}}
$$

* **Quick feel**  
  * **0**  — model no better than simply predicting the mean  
  * **0.5** — explains half the story  
  * **1**  — perfect fit  
  * **\< 0** — worse than the mean on new (test) data

#### 2.  Reading the number at a glance  

| R² value | Plain-English meaning |
|----------|----------------------|
| **≈ 0**  | No real improvement over guessing the mean. |
| **0.25** | Model explains roughly a quarter of the variability. |
| **0.50** | Captures half of what makes *y* change. |
| **≥ 0.90** | Very tight fit; only a thin slice of variation remains. |
| **\< 0** *(test data)* | Model is *worse* than the mean guess. |

> **Reminder:** Ordinary R² always climbs when you add predictors—even useless ones. Check **Adjusted R²** or cross-validated R² to be sure extra features really help.


In [ ]:
plt.figure(); sns.barplot(data=df_reg, x='Model', y='R2'); plt.xticks(rotation=45, ha='right'); plt.title('R² scores'); plt.show()

In [ ]:
best_name = df_reg.iloc[0,0]; best_est = all_models[best_name]
pred_best = best_est.predict(Xr_test)
plt.scatter(yr_test, pred_best); plt.plot([yr_test.min(), yr_test.max()],[yr_test.min(), yr_test.max()], ls='--');
plt.xlabel('True'); plt.ylabel('Predicted');
plt.title(f'Residuals – {best_name}'); plt.show()

#### Interpretation — what the numbers tell us

* **Regularised linear models (Ridge / Lasso / Elastic-Net)**  
  They keep the **straight-line assumption** of ordinary linear regression but add a penalty that **shrinks large coefficients**, cutting down variance and over-fitting. Ridge uses an L2 penalty — all weights are nudged toward zero — while Lasso can shrink some weights *exactly* to zero, performing feature selection in the process. 

* **Tree ensembles (Random Forest, Gradient-Boosting Regressor)**  
  Deep trees capture bends and feature-interactions that a straight line misses, but a single tree can be unstable. Random Forest **averages many decorrelated trees**, slashing variance, while Gradient Boosting **adds shallow trees one-after-another** to chase the residual errors. Both methods model non-linear patterns and usually beat linear models on tabular data — including this Diabetes set.

* **SVR (RBF kernel)**  
  Support-Vector Regression warps the data into a higher-dimensional space via the RBF kernel and finds the widest “tube” of acceptable error. The two knobs, **C** (penalty for errors) and **γ** (kernel width), let you trade bias vs. variance; tuned well, SVR can match or beat tree ensembles, but grid search is slower. 

* **How to read the residual plot**  
  A good residual plot looks like random scatter:  
  * **Homoscedastic** – spread stays roughly constant across predicted values.  
  * **No systematic curve** – otherwise the model missed a non-linear trend.  
  Patterns, funnels or curves signal you might need more features, non-linear terms, or a different model class.


## 4  Key Takeaways
* Always **start with baselines** – they ground expectations and give tuning a reference point.
* **Decision‑Tree pruning** (`ccp_alpha`, `min_samples_leaf`) is critical; blindly limiting depth alone may still overfit.
* **Permutation importance** is more reliable than impurity‑based importance for feature ranking in tree ensembles.
* In regression, **regularisation** balances bias‑variance, while **ensemble** and **kernel** methods capture complex patterns.
* Hyper‑parameter tuning pays off: note performance gains from tuned RandomForest, GBR and SVR compared to defaults.